# 04 · Inference demo

Three ways to run the trained detector — single image (upload / URL), video file, and the Colab webcam — plus the Gradio web app and model export.

> **Runtime:** go to *Runtime → Change runtime type → T4 GPU* before running anything.
> Every cell below is safe to re-run; nothing is lost when Colab disconnects because all
> data, checkpoints and results live on your Google Drive.

## 0.1 GPU check
**What:** prints the GPU Colab assigned to this session.
**Why:** training on CPU takes hours instead of minutes; we warn loudly if no GPU is present.

In [ ]:
import subprocess, shutil

try:
    if shutil.which("nvidia-smi") is None:
        raise FileNotFoundError("nvidia-smi not found")
    print(subprocess.check_output(["nvidia-smi"], encoding="utf-8", errors="replace"))
    GPU_AVAILABLE = True
except Exception as exc:
    GPU_AVAILABLE = False
    print("=" * 70)
    print("WARNING: No GPU detected (", exc, ")")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    print("=" * 70)

## 0.2 Mount Google Drive & get the code
**What:** mounts Drive at `/content/drive`, then either uses a copy of the repo already on Drive
or clones it from GitHub into `/content`.
**Why:** Drive is the only storage that survives a runtime disconnect. Checkpoints, the prepared
dataset and evaluation outputs are all written under `SAVE_DIR`.

Edit `REPO_URL` once (your fork), or copy the repo folder to
`MyDrive/masked-face-detection/repo` and it will be picked up automatically.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Numbu-bit/masked-face-detection.git"   # <-- change if you fork
SAVE_DIR = "/content/drive/MyDrive/masked-face-detection"                  # everything persistent lives here
DRIVE_REPO = os.path.join(SAVE_DIR, "repo")
LOCAL_REPO = "/content/masked-face-detection"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab - using the current working directory as the repo.")
    SAVE_DIR = os.path.abspath("runs")

os.makedirs(SAVE_DIR, exist_ok=True)

if IN_COLAB:
    if os.path.isfile(os.path.join(DRIVE_REPO, "configs", "default.yaml")):
        REPO_DIR = DRIVE_REPO
        print("Using repo copy on Drive:", REPO_DIR)
    else:
        REPO_DIR = LOCAL_REPO
        if not os.path.isfile(os.path.join(REPO_DIR, "configs", "default.yaml")):
            if "YOUR_USERNAME" in REPO_URL:
                raise RuntimeError(
                    "Set REPO_URL to your GitHub fork above, OR copy the repository folder to "
                    f"{DRIVE_REPO} so the notebook can find configs/default.yaml.")
            rc = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
            if rc != 0:
                raise RuntimeError(f"git clone failed for {REPO_URL}. Is the repo public / URL correct?")
        else:
            os.system(f"git -C {REPO_DIR} pull -q")
else:
    REPO_DIR = os.getcwd() if os.path.isfile("configs/default.yaml") else os.path.abspath("..")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)
print("Persistent storage:", SAVE_DIR)

## 0.3 Install dependencies
**What:** installs the tested stack from `requirements.txt`. If that cannot be installed on this
runtime's Python version, it automatically falls back to `requirements-fallback.txt`
(same libraries, range pins) and tells you so.
**Why:** exact pins avoid the "it worked yesterday" class of breakages, but Colab upgrades its
Python from time to time and old pins may have no wheels for it — the fallback keeps you running.
Takes ~1–2 minutes on a fresh runtime; instant on re-runs.

In [ ]:
import subprocess, sys, platform

print("Python", platform.python_version())


def pip_install(req_file: str) -> "subprocess.CompletedProcess":
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file],
                          capture_output=True, encoding="utf-8", errors="replace")


proc = pip_install("requirements.txt")
if proc.returncode == 0:
    print("Dependencies installed (requirements.txt).")
else:
    print("requirements.txt could not be installed on this runtime. pip said:\n")
    print(proc.stderr[-2500:])
    print("\n-> Trying requirements-fallback.txt (range pins) ...")
    proc = pip_install("requirements-fallback.txt")
    if proc.returncode == 0:
        print("Dependencies installed (requirements-fallback.txt).")
    else:
        print(proc.stderr[-2500:])
        raise RuntimeError("Both requirement sets failed. Copy the pip output above into an issue / to Claude.")

import ultralytics, torch, numpy, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| opencv", cv2.__version__, "| CUDA", torch.cuda.is_available())

## 0.4 Seeds & config
**What:** loads `configs/default.yaml` (the single source of truth for every hyper-parameter)
and seeds Python / NumPy / PyTorch / CUDA with `seed=42`.
**Why:** reproducible splits, reproducible training.

In [ ]:
import json
from src.utils import load_config, set_seed, get_device, resolve_save_dir

# Optional: MFD_OVERRIDES='{"epochs": 1}' in the environment (used by automated tests)
ENV_OVERRIDES = json.loads(os.environ.get("MFD_OVERRIDES", "{}"))
cfg = load_config(overrides={"save_dir": os.path.join(SAVE_DIR, "runs"), **ENV_OVERRIDES})
set_seed(cfg["seed"])
device = get_device()
RUN_DIR = resolve_save_dir(cfg) / cfg["run_name"]
DATA_ROOT = cfg["data_root"]
print("Model:", cfg["model_variant"], "| image size:", cfg["image_size"], "| batch:", cfg["batch_size"])
print("Run directory:", RUN_DIR)

## 1. Load the detector
**What:** wraps `best.pt` in `src.inference.Detector` (thresholds come from the config).

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPyImage
import cv2, json
from src.inference import Detector, detections_to_json
from src.utils import bgr_to_rgb, free_memory

WEIGHTS = None   # or an explicit path to best.pt
detector = Detector(cfg, weights=WEIGHTS)
OUT_DIR = Path(SAVE_DIR) / "inference_outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

def show(image_bgr, width=800):
    """Display a BGR image inline."""
    ok, buf = cv2.imencode(".jpg", image_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
    display(IPyImage(data=buf.tobytes(), width=width))

## 2A. Single image — upload or URL
**What:** upload one or more images with the Colab file picker (or set `IMAGE_URL`), run
detection, show the annotated result with the `"class 0.94"` labels and the
`Faces | Masked | Unmasked` banner, and print the JSON details.

In [ ]:
IMAGE_URL = None   # e.g. "https://upload.wikimedia.org/wikipedia/commons/…/some_face.jpg"

sources = []
if IMAGE_URL:
    sources.append(("url", IMAGE_URL))
elif IN_COLAB:
    from google.colab import files
    print("Choose one or more image files...")
    uploaded = files.upload()
    for name, data in uploaded.items():
        p = Path("/content") / name; p.write_bytes(data); sources.append(("file", p))
else:
    sources.append(("file", next(Path(DATA_ROOT, "test", "images").iterdir())))

for kind, src in sources:
    try:
        annotated, dets = detector.detect_url(src) if kind == "url" else detector.detect_file(src)
    except (FileNotFoundError, RuntimeError) as exc:
        print("Skipping", src, "->", exc); continue
    out_path = OUT_DIR / f"{Path(str(src)).stem}_detected.jpg"
    cv2.imwrite(str(out_path), annotated)
    show(annotated)
    print(json.dumps(detections_to_json(dets, cfg), indent=2))
    print("Saved:", out_path)

## 2B. Video file
**What:** upload an `.mp4` (or set `VIDEO_PATH`), annotate every frame with a progress bar, save the
result to Drive and offer it for download.
**Why:** `every_n=2` runs the model on every second frame and reuses boxes in between — roughly 2×
faster with no visible difference at 25–30 FPS; set to 1 for every frame.

In [ ]:
VIDEO_PATH = None     # e.g. "/content/drive/MyDrive/test.mp4"
EVERY_N = 2
MAX_FRAMES = None     # e.g. 300 for a quick test

if VIDEO_PATH is None and IN_COLAB:
    from google.colab import files
    print("Choose a video file...")
    up = files.upload()
    if up:
        name = next(iter(up)); VIDEO_PATH = f"/content/{name}"; Path(VIDEO_PATH).write_bytes(up[name])

if VIDEO_PATH:
    out_video = OUT_DIR / (Path(VIDEO_PATH).stem + "_detected.mp4")
    try:
        summary = detector.process_video(VIDEO_PATH, out_video, every_n=EVERY_N, max_frames=MAX_FRAMES)
    except (FileNotFoundError, RuntimeError) as exc:
        raise RuntimeError(f"Video processing failed: {exc}") from exc
    print(json.dumps(summary, indent=2))
    if IN_COLAB:
        from google.colab import files
        files.download(str(out_video))
else:
    print("No video selected - skipping.")

## 2C. Webcam (Colab) — snapshot
**What:** JavaScript opens your browser camera, you click *Capture*, the JPEG is sent to Python and
annotated.
**Why:** Colab runs in the cloud, so the only way to reach your webcam is through the browser.
Allow camera access when prompted.

In [ ]:
WEBCAM_JS = """
async function takePhoto(quality) {
  const div = document.createElement('div');
  const capture = document.createElement('button');
  capture.textContent = 'Capture';
  div.appendChild(capture);
  const video = document.createElement('video');
  video.style.display = 'block';
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  document.body.appendChild(div);
  div.appendChild(video);
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise((resolve) => capture.onclick = resolve);
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth; canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);
  stream.getVideoTracks()[0].stop();
  div.remove();
  return canvas.toDataURL('image/jpeg', quality);
}
"""

def webcam_snapshot(quality: float = 0.9) -> bytes:
    """Capture one JPEG from the browser webcam (Colab only)."""
    import base64
    from IPython.display import Javascript
    from google.colab.output import eval_js
    display(Javascript(WEBCAM_JS))
    data_url = eval_js(f"takePhoto({quality})")
    return base64.b64decode(data_url.split(",", 1)[1])

if IN_COLAB:
    try:
        jpeg = webcam_snapshot()
        annotated_jpeg, dets = detector.detect_frame_jpeg(jpeg)
        display(IPyImage(data=annotated_jpeg, width=640))
        print(json.dumps(detections_to_json(dets, cfg)["summary"]))
    except Exception as exc:
        print("Webcam capture failed:", exc, "\nAllow camera access in the browser and re-run.")
else:
    print("Webcam capture only works inside Colab.")

## 2C′. Webcam (Colab) — live stream
**What:** grabs frames continuously from the browser, runs detection on each, and redraws the
annotated frame inline. Stop with the ■ button or after `MAX_SECONDS`.
**Why:** frame round-trips go through the browser ↔ Colab bridge (~5–10 FPS); it is a demo of
live behaviour, not the model's raw throughput (see notebook 03 for that).

In [ ]:
STREAM_JS = """
var _mfd = {stream: null, video: null, canvas: null};
async function startStream() {
  if (_mfd.stream) return;
  _mfd.video = document.createElement('video');
  _mfd.stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  _mfd.video.srcObject = _mfd.stream;
  await _mfd.video.play();
  _mfd.canvas = document.createElement('canvas');
  _mfd.canvas.width = _mfd.video.videoWidth; _mfd.canvas.height = _mfd.video.videoHeight;
}
async function grabFrame(quality) {
  if (!_mfd.stream) await startStream();
  _mfd.canvas.getContext('2d').drawImage(_mfd.video, 0, 0);
  return _mfd.canvas.toDataURL('image/jpeg', quality);
}
function stopStream() {
  if (_mfd.stream) { _mfd.stream.getVideoTracks()[0].stop(); _mfd.stream = null; }
}
"""
MAX_SECONDS = 30

if IN_COLAB:
    import base64, time
    from IPython.display import Javascript, clear_output
    from google.colab.output import eval_js
    display(Javascript(STREAM_JS))
    t0, n = time.time(), 0
    try:
        while time.time() - t0 < MAX_SECONDS:
            data_url = eval_js("grabFrame(0.8)")
            jpeg = base64.b64decode(data_url.split(",", 1)[1])
            annotated_jpeg, dets = detector.detect_frame_jpeg(jpeg)
            clear_output(wait=True)
            display(IPyImage(data=annotated_jpeg, width=640))
            n += 1
            print(f"frame {n} | {n / (time.time() - t0):.1f} FPS (browser round-trip) | "
                  f"{detections_to_json(dets, cfg)['summary']}")
    except KeyboardInterrupt:
        pass
    except Exception as exc:
        print("Stream failed:", exc)
    finally:
        eval_js("stopStream()")
        print("Stream stopped.")
else:
    print("Live webcam only works inside Colab.")

## 3. Gradio web app (public share link)
**What:** launches `demo/gradio_app.py` inside the notebook with `share=True`; Gradio prints a
`https://xxxx.gradio.live` URL you can open on any device (valid ~72 h while this cell runs).
The app has two tabs: **Image / snapshot** (upload a file or take a webcam photo) and
**Live webcam** (continuous detection on the camera stream). The browser asks for camera
permission the first time you use either webcam feature.
**Why:** the easiest way to let someone else try the model without touching code.
Interrupt the cell to stop the server.

In [ ]:
from demo.gradio_app import build_demo

free_memory()
demo = build_demo(cfg, weights=WEIGHTS)
try:
    # In Colab the call blocks so the public link stays alive; locally it returns immediately.
    demo.launch(share=IN_COLAB, debug=False, show_error=True, prevent_thread_lock=not IN_COLAB)
except (ValueError, OSError) as exc:
    print("Gradio could not start a server here:", exc)
    print("In Colab this cell prints a public https://xxxx.gradio.live link. "
          "Locally, run:  python demo/gradio_app.py --share")

## 4. Export for deployment (ONNX · TFLite · TorchScript)
**What:** runs `scripts/export_model.py`, which exports `best.pt` and validates each format on 5
test images against the PyTorch model (boxes must agree with IoU > 0.99).
**Why:** ONNX runs anywhere (onnxruntime), TFLite targets phones, TorchScript is for PyTorch serving.
TFLite pulls in TensorFlow (~2–4 min install) and is attempted last; the other two never depend on it.
Exported files land next to `best.pt` on Drive together with `export_report.json`.

In [ ]:
import subprocess, sys

demo.close() if "demo" in globals() else None
free_memory()
from src.utils import find_best_checkpoint, save_config

EXPORT_FORMATS = None   # e.g. ["onnx", "torchscript"] to skip the slow TFLite/TensorFlow install
EXPORT_FORMATS = EXPORT_FORMATS or os.environ.get("MFD_EXPORT_FORMATS", "").split() or None

weights = Path(WEIGHTS) if WEIGHTS else find_best_checkpoint(cfg)
if weights is None or not weights.exists():
    raise FileNotFoundError("best.pt not found - train in notebook 02 first or set WEIGHTS above.")
# The script runs in a subprocess, so hand it this session's config (thresholds, image size, ...)
cfg_snapshot = save_config(cfg, weights.parent / "export_config.yaml")

cmd = [sys.executable, "scripts/export_model.py", "--weights", str(weights),
       "--config", str(cfg_snapshot), "--data-root", DATA_ROOT]
if EXPORT_FORMATS:
    cmd += ["--formats", *EXPORT_FORMATS]
proc = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
print(proc.stdout[-6000:]); print(proc.stderr[-3000:])
if proc.returncode != 0:
    print("Export reported problems - see above (TFLite issues do not affect ONNX/TorchScript).")

## 5. Export the model for the web app (Render deployment)
**What:** exports `best.pt` to `model.onnx` at **416 px** (fast enough to run live in a browser via
onnxruntime-web), saves it to `MyDrive/masked-face-detection/model.onnx` and offers it for download.
This cell only needs the setup cells (0.1–0.4) — you can skip everything else in this notebook.
**Why:** the deployed app (`web/`) runs the model inside the visitor's browser; Render's free tier has
no GPU. Put the downloaded file at `web/models/model.onnx` in the repo and push (or upload it as a
GitHub Release and set `MODEL_URL` on Render). See `web/README.md`.
Use `WEB_IMGSZ = 640` for maximum accuracy at the cost of ~2.4x slower frames.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

from src.utils import find_best_checkpoint, save_config

WEB_IMGSZ = 416
WEIGHTS = globals().get("WEIGHTS")           # set in section 1 if you ran it; else best.pt from the run dir
weights = Path(WEIGHTS) if WEIGHTS else find_best_checkpoint(cfg)
if weights is None or not weights.exists():
    raise FileNotFoundError("best.pt not found - train in notebook 02 first or set WEIGHTS.")
cfg_snapshot = save_config(cfg, weights.parent / "export_config.yaml")
web_model = Path(SAVE_DIR) / "model.onnx"
cmd = [sys.executable, "scripts/prepare_web_model.py", "--weights", str(weights), "--config", str(cfg_snapshot),
       "--imgsz", str(WEB_IMGSZ), "--dest", str(web_model)]
proc = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
print(proc.stdout[-3000:]); print(proc.stderr[-1500:])
if proc.returncode != 0 or not web_model.exists():
    raise RuntimeError("Web export failed - see above.")
print(f"Saved {web_model} ({web_model.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(web_model))
print("Next: copy this file to web/models/model.onnx in the repo, commit, push -> Render deploys it.")